In [30]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import re

options = webdriver.ChromeOptions()
# Uncomment the next line to run headless (no browser window)
# options.add_argument('--headless')

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

url = "https://www.tender247.com/tenders?state=delhi&advancesearch=true"
driver.get(url)
time.sleep(4)
# Scroll to load more tenders
for _ in range(25):  # Increase range if you need more tenders
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)

cards = driver.find_elements(By.CSS_SELECTOR, 'div.w-\\[100\\%\\].float-left')
print(f"Found {len(cards)} cards, parsing...")

data = []

for card in cards:
    card_html = card.get_attribute('innerHTML')
    if "Bid Value:" not in card_html or "T247 ID-" not in card_html:
        continue

    try:
        # Bid Value
        try:
            bid_value = card.find_element(By.XPATH, ".//span[contains(text(),'Bid Value:')]/following-sibling::div").text.strip()
        except:
            bid_value = ""
        # EMD
        try:
            emd = card.find_element(By.XPATH, ".//span[contains(text(),'EMD:')]/following-sibling::div").text.strip()
        except:
            emd = ""
        # Closing Date & Days Left (robust version)
        try:
            closing_date = ""
            days_left = ""
            # Find all relevant float-left mr-2 divs in this card
            all_float_divs = card.find_elements(By.XPATH, ".//div[contains(@class, 'float-left') and contains(@class, 'mr-2')]")
            for fd in all_float_divs:
                m = re.match(r"\s*([0-9]{2}-[0-9]{2}-[0-9]{4})", fd.text)
                if m:
                    closing_date = m.group(1)
                    try:
                        days_left_span = fd.find_element(By.TAG_NAME, "span")
                        days_left = days_left_span.text.strip()
                    except:
                        days_left = ""
                    break
        except Exception as e:
            closing_date = ""
            days_left = ""
        # T247 ID
        try:
            t247_id = card.find_element(By.XPATH, ".//span[contains(text(),'T247 ID-')]/..").text.replace("T247 ID-","").replace("|","").strip()
        except:
            t247_id = ""
        # Description
        try:
            description = card.find_element(By.XPATH, ".//p/a/span").text.strip()
        except:
            description = ""
        # Location (robust: extract after SVG)
        try:
            all_h3s = card.find_elements(By.TAG_NAME, "h3")
            location = ""
            for h3 in all_h3s:
                svgs = h3.find_elements(By.TAG_NAME, "svg")
                if svgs and "India" in h3.text:
                    inner_html = h3.get_attribute('innerHTML')
                    if "</svg>" in inner_html:
                        location = inner_html.split("</svg>")[-1].strip()
                    else:
                        location = h3.text.strip()
                    break
        except:
            location = ""
        # Bid Link
        try:
            bid_link = card.find_element(By.XPATH, ".//a[contains(@href, '/tender-details/')]").get_attribute("href")
            bid_link = "https://www.tender247.com" + bid_link if bid_link.startswith("/") else bid_link
        except:
            bid_link = ""

        data.append({
            "T247 ID": t247_id,
            "Bid Value": bid_value,
            "EMD": emd,
            "Closing Date": closing_date,
            "Days Left": days_left,
            "Description": description,
            "Location": location,
            "Bid Link": bid_link,
        })

    except Exception as e:
        print("Error on card:", e)
        continue

driver.quit()

df = pd.DataFrame(data)
df.to_excel("Tender247_delhi.xlsx", index=False)
print("Scraping done! File: Tender247_delhi.xlsx")


Found 322 cards, parsing...
Scraping done! File: Tender247_delhi.xlsx
